In [1]:
# CELL 0 - Environment check and installs
!nvidia-smi || echo "No GPU detected (OK, will use CPU)."

# Install packages
import sys
!pip install -q ultralytics==8.1.23  # or latest stable if you prefer
!pip install -q ensemble-boxes pycocotools tqdm

import os
if not os.path.exists('/content/yolov5'):
    !git clone -q https://github.com/ultralytics/yolov5.git /content/yolov5
    %cd /content/yolov5
    !pip install -q -r requirements.txt
    %cd -
print("Install step done.")

/bin/bash: line 1: nvidia-smi: command not found
No GPU detected (OK, will use CPU).
Install step done.


In [2]:
# CELL 1 - show folder tree & image counts under /content/datasets
import os, glob
ROOT = '/content/datasets'
if not os.path.exists(ROOT):
    print("No /content/datasets folder exists. Upload your dataset to /content/datasets or mount Drive.")
else:
    print("Top-level items in /content/datasets:")
    for name in sorted(os.listdir(ROOT)):
        path = os.path.join(ROOT, name)
        if os.path.isdir(path):
            imgs = glob.glob(os.path.join(path, '**', '*.*'), recursive=True)
            img_count = sum(1 for f in imgs if f.lower().endswith(('.jpg','.jpeg','.png','.bmp')))
            print(f" - {name:30s}  | images (recursively): {img_count}")
        else:
            print(" - (file) ", name)
print("\nIf your images are not under /content/datasets, upload them now (zip is easiest).")

Top-level items in /content/datasets:
 - (file)  LICENSE.txt
 - (file)  african-wildlife.yaml
 - (file)  african-wildlife.zip
 - african_small                   | images (recursively): 250
 - images                          | images (recursively): 1504
 - labels                          | images (recursively): 0

If your images are not under /content/datasets, upload them now (zip is easiest).


In [3]:
# CELL 2 - Build subset into /content/datasets/african_small (preserve classes if present)
import os, glob, random, shutil
ROOT = '/content/datasets'
SUBSET = os.path.join(ROOT, 'african_small')
TRAIN_OUT = os.path.join(SUBSET, 'images', 'train')
VAL_OUT = os.path.join(SUBSET, 'images', 'val')

MAX_IMAGES = 250        # cap total images (change lower if slow)
MAX_PER_CLASS = 200     # cap per class if classes exist
RSEED = 42
random.seed(RSEED)

# collect candidate image folders and classes
def find_class_folders(root):
    classes = []
    for entry in sorted(os.listdir(root)):
        full = os.path.join(root, entry)
        if os.path.isdir(full):
            # check if folder directly contains images
            imgs = [f for f in os.listdir(full) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]
            if imgs:
                classes.append(entry)
    return classes

classes = find_class_folders(ROOT)
# If no direct class folders at top, try deeper one level
if not classes:
    for d in sorted(os.listdir(ROOT)):
        p = os.path.join(ROOT, d)
        if os.path.isdir(p):
            inner = find_class_folders(p)
            if inner:
                classes = inner
                # remap ROOT to that folder for searching images
                ROOT = p
                print("Using nested folder", p, "as dataset root")
                break

print("Detected classes (if any):", classes)

# Clear & create dest
shutil.rmtree(SUBSET, ignore_errors=True)
os.makedirs(TRAIN_OUT, exist_ok=True)
os.makedirs(VAL_OUT, exist_ok=True)

# If class folders exist under ROOT, preserve them
total_copied = 0
if classes:
    print("Preserving class subfolders. Copying up to MAX_PER_CLASS per class until MAX_IMAGES total reached.")
    for cls in classes:
        src_dir = os.path.join(ROOT, cls)
        imgs = sorted([os.path.join(src_dir,f) for f in os.listdir(src_dir) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])
        if not imgs:
            continue
        # sample up to MAX_PER_CLASS
        selection = imgs[:MAX_PER_CLASS]
        # split 80/20
        split = int(0.8 * len(selection))
        train_imgs = selection[:split]
        val_imgs = selection[split:]
        # copy into train/val with class subfolder
        for s in train_imgs:
            dst_dir = os.path.join(TRAIN_OUT, cls); os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(s, os.path.join(dst_dir, os.path.basename(s))); total_copied += 1
            if total_copied >= MAX_IMAGES: break
        if total_copied >= MAX_IMAGES: break
        for s in val_imgs:
            dst_dir = os.path.join(VAL_OUT, cls); os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(s, os.path.join(dst_dir, os.path.basename(s))); total_copied += 1
            if total_copied >= MAX_IMAGES: break
        if total_copied >= MAX_IMAGES: break
else:
    # flat copy: find images anywhere under original root (excluding african_small itself)
    candidates = []
    for root, dirs, files in os.walk(ROOT):
        if 'african_small' in root.split(os.sep): continue
        for f in files:
            if f.lower().endswith(('.jpg','.jpeg','.png','.bmp')):
                candidates.append(os.path.join(root,f))
    candidates = sorted(set(candidates))
    if not candidates:
        raise RuntimeError(f"No images found under {ROOT}. Upload images to /content/datasets and re-run.")
    # sample up to MAX_IMAGES
    if len(candidates) > MAX_IMAGES:
        candidates = random.sample(candidates, MAX_IMAGES)
    split = int(0.8 * len(candidates))
    train_imgs = candidates[:split]; val_imgs = candidates[split:]
    for s in train_imgs:
        shutil.copy2(s, os.path.join(TRAIN_OUT, os.path.basename(s))); total_copied += 1
    for s in val_imgs:
        shutil.copy2(s, os.path.join(VAL_OUT, os.path.basename(s))); total_copied += 1

print(f"Copied total {total_copied} images into subset {SUBSET}")
print("Train sample files:", next(os.walk(TRAIN_OUT))[1] if os.path.isdir(TRAIN_OUT) else os.listdir(TRAIN_OUT)[:5])

Using nested folder /content/datasets/images as dataset root
Detected classes (if any): ['test', 'train', 'val']
Preserving class subfolders. Copying up to MAX_PER_CLASS per class until MAX_IMAGES total reached.
Copied total 250 images into subset /content/datasets/african_small
Train sample files: ['train', 'test']


In [4]:
# CELL 3 - Create data_custom.yaml using the real dataset yaml
import os, yaml

SUBSET = '/content/datasets/african_small'
train_path = os.path.join(SUBSET, 'images', 'train')
val_path   = os.path.join(SUBSET, 'images', 'val')

# Use the real yaml shipped with the dataset if available
real_yaml = '/content/datasets/african-wildlife.yaml'
if os.path.exists(real_yaml):
    with open(real_yaml) as f:
        src = yaml.safe_load(f)
    names = src.get('names', [])
    print("Loaded class names from dataset yaml:", names)
else:
    # Fallback: known african-wildlife classes
    names = ['buffalo', 'elephant', 'rhino', 'zebra']
    print("Using hardcoded class names:", names)

data = {
    'train': train_path,
    'val':   val_path,
    'nc':    len(names),
    'names': names
}
yaml_path = os.path.join(SUBSET, 'data_custom.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("Wrote dataset YAML:", yaml_path)
print(data)

Loaded class names from dataset yaml: {0: 'buffalo', 1: 'elephant', 2: 'rhino', 3: 'zebra'}
Wrote dataset YAML: /content/datasets/african_small/data_custom.yaml
{'train': '/content/datasets/african_small/images/train', 'val': '/content/datasets/african_small/images/val', 'nc': 4, 'names': {0: 'buffalo', 1: 'elephant', 2: 'rhino', 3: 'zebra'}}


In [5]:
# CELL 4 - Copy real YOLO labels (preserving class subfolders) OR generate pseudo-labels
import os, glob, shutil
from pathlib import Path
from PIL import Image  # import FIRST

SUBSET = Path('/content/datasets/african_small')

# Mirror the image subfolder structure in labels/
for split in ['train', 'val']:
    img_root = SUBSET / 'images' / split
    lbl_root = SUBSET / 'labels' / split
    shutil.rmtree(lbl_root, ignore_errors=True)

    for img_path in sorted(img_root.rglob('*.*')):
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png', '.bmp'):
            continue
        # Preserve subfolder structure (e.g., labels/train/zebra/)
        rel = img_path.relative_to(img_root)
        lbl_path = lbl_root / rel.parent / (img_path.stem + '.txt')
        lbl_path.parent.mkdir(parents=True, exist_ok=True)

        # Try to find a matching real label anywhere under /content/datasets
        # (excluding our subset) by filename stem
        matches = [
            p for p in glob.glob(f'/content/datasets/**/{img_path.stem}.txt', recursive=True)
            if 'african_small' not in p
        ]
        if matches:
            shutil.copy2(matches[0], lbl_path)
        else:
            lbl_path.write_text('')  # placeholder

# Count non-empty labels copied
real_labels = [
    p for p in SUBSET.rglob('labels/**/*.txt') if os.path.getsize(p) > 0
]
print(f"Non-empty label files found: {len(real_labels)}")

# ── Pseudo-label only images that still have empty labels ──────────────────
GENERATE_PSEUDO = True
CONF = 0.35

# Load yaml to get nc for class remapping
import yaml
with open(str(SUBSET / 'data_custom.yaml')) as f:
    data_cfg = yaml.safe_load(f)
NC = data_cfg['nc']

if GENERATE_PSEUDO:
    empty_labels = [
        p for p in SUBSET.rglob('labels/**/*.txt') if os.path.getsize(p) == 0
    ]
    if not empty_labels:
        print("All labels already populated — skipping pseudo-labeling.")
    else:
        print(f"Generating pseudo-labels for {len(empty_labels)} unlabelled images...")
        from ultralytics import YOLO
        model = YOLO('yolov8n.pt')
        total_boxes = 0

        def preds_to_yolo_txt(res, img_path, out_txt_path, conf_thr, nc):
            """Write YOLO-format label; remap class IDs to [0, nc-1]."""
            W, H = Image.open(img_path).size
            lines = []
            boxes = getattr(res, 'boxes', None) or []
            for b in boxes:
                conf = float(b.conf.cpu().numpy()[0])
                if conf < conf_thr:
                    continue
                raw_cls = int(b.cls.cpu().numpy()[0])
                cls = raw_cls % nc          # remap COCO id → valid range
                xyxy = b.xyxy.cpu().numpy()[0]
                x1, y1, x2, y2 = xyxy
                xc = (x1 + x2) / (2 * W)
                yc = (y1 + y2) / (2 * H)
                bw = (x2 - x1) / W
                bh = (y2 - y1) / H
                lines.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
            Path(out_txt_path).write_text("\n".join(lines))
            return len(lines)

        for lbl_path in empty_labels:
            # Reconstruct matching image path
            lbl_path = Path(lbl_path)
            rel = lbl_path.relative_to(SUBSET / 'labels')   # e.g. train/zebra/img.txt
            img_path = SUBSET / 'images' / rel.parent / (lbl_path.stem + '.jpg')
            if not img_path.exists():
                # Try other extensions
                for ext in ('.png', '.jpeg', '.bmp'):
                    candidate = img_path.with_suffix(ext)
                    if candidate.exists():
                        img_path = candidate
                        break
            if not img_path.exists():
                continue
            res = model.predict(source=str(img_path), imgsz=640, conf=CONF, verbose=False)[0]
            n = preds_to_yolo_txt(res, str(img_path), str(lbl_path), CONF, NC)
            total_boxes += n

        print(f"Pseudo-labeling done. Total boxes written: {total_boxes}")
else:
    print("Skipping pseudo-label generation.")

Non-empty label files found: 250
All labels already populated — skipping pseudo-labeling.


In [6]:
# CELL 5 - Verify images & labels with subfolder-aware counts
import yaml, glob, os

yaml_path = '/content/datasets/african_small/data_custom.yaml'
with open(yaml_path) as f:
    data = yaml.safe_load(f)
print("YAML:", data)

def count_images(path):
    return len([p for p in glob.glob(os.path.join(path,'**','*.*'), recursive=True)
                if p.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])

def count_labels(path, non_empty=False):
    txts = glob.glob(os.path.join(path,'**','*.txt'), recursive=True)
    if non_empty:
        return sum(1 for t in txts if os.path.getsize(t) > 0)
    return len(txts)

print(f"\nTrain images : {count_images(data['train'])}")
print(f"Val   images : {count_images(data['val'])}")

lbl_train = '/content/datasets/african_small/labels/train'
lbl_val   = '/content/datasets/african_small/labels/val'
print(f"\nTrain labels (total)     : {count_labels(lbl_train)}")
print(f"Train labels (non-empty) : {count_labels(lbl_train, non_empty=True)}")
print(f"Val   labels (total)     : {count_labels(lbl_val)}")
print(f"Val   labels (non-empty) : {count_labels(lbl_val, non_empty=True)}")

# Show a non-empty example
for l in glob.glob(os.path.join(lbl_val,'**','*.txt'), recursive=True):
    if os.path.getsize(l) > 0:
        print(f"\nExample label ({l}):\n{open(l).read()}")
        break

print("\nIf non-empty label counts > 0, training should work. Re-run CELL 6.")

YAML: {'names': {0: 'buffalo', 1: 'elephant', 2: 'rhino', 3: 'zebra'}, 'nc': 4, 'train': '/content/datasets/african_small/images/train', 'val': '/content/datasets/african_small/images/val'}

Train images : 210
Val   images : 40

Train labels (total)     : 210
Train labels (non-empty) : 210
Val   labels (total)     : 40
Val   labels (non-empty) : 40

Example label (/content/datasets/african_small/labels/val/test/4 (179).txt):
3 0.542969 0.629167 0.235937 0.595833
3 0.654687 0.446875 0.381250 0.477083
3 0.385937 0.416667 0.315625 0.516667


If non-empty label counts > 0, training should work. Re-run CELL 6.


In [8]:
import torch
import ultralytics.nn.tasks as ult_tasks
import ultralytics.nn.modules as ult_modules

# PyTorch 2.6 changed weights_only=True by default, which blocks Ultralytics globals.
# We need to allowlist the required Ultralytics classes.

safe_classes = [
    ult_tasks.DetectionModel,
    ult_tasks.SegmentationModel,
    ult_tasks.ClassificationModel,
    ult_tasks.PoseModel,
]

# Add each class to PyTorch's safe globals list
for cls in safe_classes:
    try:
        torch.serialization.add_safe_globals([cls])
    except Exception as e:
        print(f"Could not add {cls}: {e}")

# Also monkey-patch torch.load to force weights_only=False for .pt files
# (safe since we are loading from ultralytics official weights)
_original_torch_load = torch.load

def _patched_torch_load(f, *args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _original_torch_load(f, *args, **kwargs)

torch.load = _patched_torch_load

print("PyTorch version   :", torch.__version__)
print("Torch.load patched: weights_only now defaults to False")
print("Safe globals registered for Ultralytics model classes.")

PyTorch version   : 2.10.0+cpu
Torch.load patched: weights_only now defaults to False
Safe globals registered for Ultralytics model classes.


In [9]:
# CELL 6 - YOLOv8 Training with validation image saving
from ultralytics import YOLO
import shutil, glob, os

data_yaml_local = '/content/datasets/african_small/data_custom.yaml'
print("Training YOLOv8 on:", data_yaml_local)

model_v8 = YOLO("yolov8n.pt")
results_v8 = model_v8.train(
    data=data_yaml_local,
    epochs=6,
    imgsz=640,
    batch=8,
    workers=2,
    patience=0,
    save=True,
    name='yolov8_african'
)

# Save best weights
v8_candidates = glob.glob('runs/detect/yolov8_african*/weights/best.pt')
if v8_candidates:
    shutil.copy(v8_candidates[-1], '/content/yolov8_best.pt')
    print("Saved: /content/yolov8_best.pt")
else:
    print("WARNING: yolov8 best.pt not found")

# Run validation and save images
model_v8_best = YOLO('/content/yolov8_best.pt')
val_results_v8 = model_v8_best.val(
    data=data_yaml_local,
    imgsz=640,
    save=True,
    name='yolov8_val'
)

# Copy validation images to organized output folder
os.makedirs('/content/outputs/yolov8/val_images', exist_ok=True)
for img in glob.glob('runs/detect/yolov8_val*/**/*.jpg', recursive=True):
    shutil.copy(img, '/content/outputs/yolov8/val_images/')
for img in glob.glob('runs/detect/yolov8_val*/**/*.png', recursive=True):
    shutil.copy(img, '/content/outputs/yolov8/val_images/')

print(f"YOLOv8 mAP50: {val_results_v8.box.map50:.4f}")
print(f"YOLOv8 mAP50-95: {val_results_v8.box.map:.4f}")
print("Validation images saved to /content/outputs/yolov8/val_images/")

Training YOLOv8 on: /content/datasets/african_small/data_custom.yaml
New https://pypi.org/project/ultralytics/8.4.17 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.1.23 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon 2.20GHz)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/datasets/african_small/data_custom.yaml, epochs=6, time=None, patience=0, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=2, project=None, name=yolov8_african, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


...

                   all         40         72      0.444      0.333      0.548      0.395
              elephant         40          1      0.332          1      0.995      0.796
                 rhino         40          1          0          0     0.0293     0.0205
                 zebra         40         70          1          0      0.621      0.369
Speed: 2.4ms preprocess, 205.7ms inference, 0.0ms loss, 6.0ms postprocess per image
Results saved to runs/detect/yolov8_african


lr/pg0,▁▆█▇▇▃
lr/pg1,▁▆█▇▇▃
lr/pg2,▁▆█▇▇▃
metrics/mAP50(B),▂▁▁▅▂█
metrics/mAP50-95(B),▂▁▁▅▁█
metrics/precision(B),▁▁▆█▇█
metrics/recall(B),██▅▄▅▁
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


Saved: /content/yolov8_best.pt
Ultralytics YOLOv8.1.23 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon 2.20GHz)
Model summary (fused): 169 layers, 3006428 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /content/datasets/african_small/labels/val/test.cache... 40 images, 0 backgrounds, 0 corrupt: 100%|██████████| 40/40 [00:00<?, ?it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:08<00:00,  2.91s/it]


                   all         40         72    0.00661      0.638      0.216      0.153
              elephant         40          1          0          0          0          0
                 rhino         40          1   0.000238          1     0.0172     0.0172
                 zebra         40         70     0.0196      0.914      0.629      0.442
Speed: 4.0ms preprocess, 195.6ms inference, 0.0ms loss, 5.1ms postprocess per image
Results saved to runs/detect/yolov8_val
YOLOv8 mAP50: 0.2155
YOLOv8 mAP50-95: 0.1530
Validation images saved to /content/outputs/yolov8/val_images/


In [19]:
# CELL 7 - YOLOv5 Training
import os, shutil, glob, subprocess

data_yaml_local = '/content/datasets/african_small/data_custom.yaml'

# Clone YOLOv5 if not already present
if not os.path.exists('/content/yolov5'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ultralytics/yolov5.git', '/content/yolov5'])
    subprocess.run(['pip', 'install', '-q', '-r', '/content/yolov5/requirements.txt'])

os.makedirs('/content/outputs/yolov5/val_images', exist_ok=True)

# Train YOLOv5n
%cd /content/yolov5
!python train.py \
    --img 640 \
    --batch 8 \
    --epochs 6 \
    --data {data_yaml_local} \
    --weights yolov5n.pt \
    --name yolov5_african \
    --patience 0 \
    --save-period 1

%cd /content

# Save best weights
v5_candidates = glob.glob('/content/yolov5/runs/train/yolov5_african*/weights/best.pt')
if v5_candidates:
    shutil.copy(v5_candidates[-1], '/content/yolov5_best.pt')
    print("Saved: /content/yolov5_best.pt")
else:
    print("WARNING: yolov5 best.pt not found")

# Run validation and save images
%cd /content/yolov5
!python val.py \
    --img 640 \
    --data {data_yaml_local} \
    --weights /content/yolov5_best.pt \
    --name yolov5_val \
    --save-txt \
    --save-conf

%cd /content

# Copy validation images
for img in glob.glob('/content/yolov5/runs/val/yolov5_val*/**/*.jpg', recursive=True):
    shutil.copy(img, '/content/outputs/yolov5/val_images/')
for img in glob.glob('/content/yolov5/runs/val/yolov5_val*/**/*.png', recursive=True):
    shutil.copy(img, '/content/outputs/yolov5/val_images/')

print("YOLOv5 validation images saved to /content/outputs/yolov5/val_images/")

# Parse YOLOv5 metrics from results.csv
import pandas as pd
results_csv = glob.glob('/content/yolov5/runs/train/yolov5_african*/results.csv')
if results_csv:
    df = pd.read_csv(results_csv[-1])
    df.columns = df.columns.str.strip()
    last = df.iloc[-1]
    v5_map50 = float(last['metrics/mAP50(B)']) if 'metrics/mAP50(B)' in df.columns else 0.0
    v5_map5095 = float(last['metrics/mAP50-95(B)']) if 'metrics/mAP50-95(B)' in df.columns else 0.0
    print(f"YOLOv5 mAP50: {v5_map50:.4f}")
    print(f"YOLOv5 mAP50-95: {v5_map5095:.4f}")
else:
    v5_map50, v5_map5095 = 0.0, 0.0
    print("Could not find YOLOv5 results.csv")


# Add to end of Cell 7 — save weights outside yolov5 folder immediately
import glob, shutil
v5_candidates = glob.glob('/content/yolov5/runs/train/yolov5_african*/weights/best.pt')
if v5_candidates:
    shutil.copy(v5_candidates[-1], '/content/yolov5_best.pt')
    print("Saved: /content/yolov5_best.pt")
else:
    print("WARNING: yolov5 best.pt not found — check Cell 7 training completed.")

/content/yolov5
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2026-02-26 18:00:42.296268: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772128842.359163   38313 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772128842.378312   38313 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772128842.429483   38313 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772128842.429553   38313 computation_placer.cc:177] computation placer a

In [20]:
# CELL 8 - Ensemble Learning using Weighted Boxes Fusion (WBF)
import os, glob, shutil, sys
import numpy as np
from PIL import Image
from ensemble_boxes import weighted_boxes_fusion
import torch
import yaml

os.makedirs('/content/outputs/ensemble/val_images', exist_ok=True)
os.makedirs('/content/outputs/ensemble/test_images', exist_ok=True)

# ── Step 1: Back up weights BEFORE re-cloning ─────────────────────────────────
# The re-clone deletes /content/yolov5 including any trained weights inside it.
# So we copy them out first.
v5_weight_src = glob.glob('/content/yolov5/runs/train/**/weights/best.pt', recursive=True)
if not os.path.exists('/content/yolov5_best.pt'):
    if v5_weight_src:
        shutil.copy(v5_weight_src[-1], '/content/yolov5_best.pt')
        print(f"Backed up YOLOv5 weights to /content/yolov5_best.pt")
    else:
        raise FileNotFoundError(
            "No YOLOv5 weights found. "
            "Please re-run Cell 7 (YOLOv5 training) before running this cell."
        )
else:
    print("YOLOv5 weights already at /content/yolov5_best.pt — skipping backup.")

# ── Step 2: Re-clone YOLOv5 fresh ────────────────────────────────────────────
print("Re-cloning YOLOv5...")
shutil.rmtree('/content/yolov5', ignore_errors=True)
os.system('git clone -q https://github.com/ultralytics/yolov5.git /content/yolov5')

# Filter ultralytics out of requirements to prevent version downgrade
with open('/content/yolov5/requirements.txt', 'r') as f:
    reqs = f.readlines()
filtered = [r for r in reqs if 'ultralytics' not in r.lower()]
with open('/content/yolov5/requirements_filtered.txt', 'w') as f:
    f.writelines(filtered)
os.system('pip install -q -r /content/yolov5/requirements_filtered.txt')
os.system('pip install -q -U "ultralytics>=8.3.0"')
print("YOLOv5 re-cloned and deps installed.")

# ── Step 3: Patch general.py ──────────────────────────────────────────────────
general_py = '/content/yolov5/utils/general.py'
with open(general_py, 'r') as f:
    lines = f.readlines()

broken  = 'from ultralytics.utils.patches import torch_load'
patched = 'torch_load = torch.load  # patched: torch_load removed from ultralytics\n'

new_lines = []
patch_applied = False
for line in lines:
    if line.strip() == broken:
        new_lines.append(patched)
        patch_applied = True
    else:
        new_lines.append(line)

with open(general_py, 'w') as f:
    f.writelines(new_lines)
print(f"Patch applied: {patch_applied}")

# ── Step 4: Clear stale module cache ─────────────────────────────────────────
to_delete = [k for k in sys.modules.keys()
             if any(x in k for x in [
                 'models', 'utils.general', 'utils.augment',
                 'utils.torch_utils', 'utils.dataloaders',
                 'utils.loss', 'utils.metrics', 'utils.plots',
                 'ultralytics'
             ])]
for k in to_delete:
    del sys.modules[k]
print(f"Cleared {len(to_delete)} cached modules.")

# ── Step 5: Add yolov5 to path and import ────────────────────────────────────
if '/content/yolov5' in sys.path:
    sys.path.remove('/content/yolov5')
sys.path.insert(0, '/content/yolov5')

from models.common import DetectMultiBackend
from utils.general import non_max_suppression, scale_boxes
from utils.augmentations import letterbox
print("YOLOv5 imports successful.")

# ── Step 6: Load both models ──────────────────────────────────────────────────
from ultralytics import YOLO
model_v8 = YOLO('/content/yolov8_best.pt')
print("YOLOv8 loaded.")

def load_yolov5_local(weights_path, device='cpu'):
    model = DetectMultiBackend(weights_path, device=torch.device(device))
    model.eval()
    return model

model_v5_local = load_yolov5_local('/content/yolov5_best.pt')
print("YOLOv5 loaded.")

# ── Step 7: Inference helpers ─────────────────────────────────────────────────
def yolov5_predict(model, img_path, conf_thr=0.25, iou_thr=0.45, imgsz=640):
    img0 = np.array(Image.open(img_path).convert('RGB'))
    H0, W0 = img0.shape[:2]
    img = letterbox(img0, imgsz, stride=int(model.stride), auto=True)[0]
    img = img.transpose((2, 0, 1))[::-1]
    img = np.ascontiguousarray(img)
    img_tensor = torch.from_numpy(img).float() / 255.0
    img_tensor = img_tensor.unsqueeze(0)
    with torch.no_grad():
        pred = model(img_tensor)
        pred = non_max_suppression(pred, conf_thr, iou_thr)
    boxes, scores, labels = [], [], []
    if pred[0] is not None and len(pred[0]):
        det = pred[0].clone()
        det[:, :4] = scale_boxes(img_tensor.shape[2:], det[:, :4], img0.shape).round()
        for *xyxy, conf, cls in det.tolist():
            x1, y1, x2, y2 = xyxy
            boxes.append([max(0,x1/W0), max(0,y1/H0), min(1,x2/W0), min(1,y2/H0)])
            scores.append(float(conf))
            labels.append(int(cls))
    return boxes, scores, labels

# ── Step 8: Ensemble config ───────────────────────────────────────────────────
IOU_THR  = 0.5
SKIP_BOX = 0.0001
CONF_THR = 0.25
WEIGHTS  = [1, 1]

with open('/content/datasets/african_small/data_custom.yaml') as f:
    cfg = yaml.safe_load(f)
class_names = cfg['names']

def run_ensemble_on_image(img_path):
    img  = Image.open(img_path).convert('RGB')
    W, H = img.size
    all_boxes, all_scores, all_labels = [], [], []

    # YOLOv8 predictions
    res_v8 = model_v8.predict(source=img_path, imgsz=640, conf=CONF_THR, verbose=False)[0]
    b8, s8, l8 = [], [], []
    if res_v8.boxes is not None:
        for b in res_v8.boxes:
            x1,y1,x2,y2 = b.xyxy.cpu().numpy()[0]
            b8.append([max(0,x1/W), max(0,y1/H), min(1,x2/W), min(1,y2/H)])
            s8.append(float(b.conf.cpu().numpy()[0]))
            l8.append(int(b.cls.cpu().numpy()[0]))
    all_boxes.append(b8 if b8 else [[0,0,0,0]])
    all_scores.append(s8 if s8 else [0.0])
    all_labels.append(l8 if l8 else [0])

    # YOLOv5 predictions
    b5, s5, l5 = yolov5_predict(model_v5_local, img_path, CONF_THR)
    all_boxes.append(b5 if b5 else [[0,0,0,0]])
    all_scores.append(s5 if s5 else [0.0])
    all_labels.append(l5 if l5 else [0])

    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        all_boxes, all_scores, all_labels,
        weights=WEIGHTS, iou_thr=IOU_THR, skip_box_thr=SKIP_BOX
    )
    return fused_boxes, fused_scores, fused_labels, W, H


def draw_and_save(img_path, fused_boxes, fused_scores,
                  fused_labels, W, H, out_path, class_names):
    import cv2
    img_cv = cv2.imread(img_path)
    if img_cv is None:
        img_cv = cv2.cvtColor(
            np.array(Image.open(img_path).convert('RGB')), cv2.COLOR_RGB2BGR)
    for box, score, label in zip(fused_boxes, fused_scores, fused_labels):
        x1=int(box[0]*W); y1=int(box[1]*H)
        x2=int(box[2]*W); y2=int(box[3]*H)
        cls_name = (class_names[int(label)]
                    if int(label) < len(class_names) else str(int(label)))
        cv2.rectangle(img_cv, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(img_cv, f"{cls_name} {score:.2f}",
                    (x1, max(y1-5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
    cv2.imwrite(out_path, img_cv)


# ── Step 9: Run ensemble on VAL images ───────────────────────────────────────
val_images = glob.glob(
    '/content/datasets/african_small/images/val/**/*.*', recursive=True)
val_images = [p for p in val_images
              if p.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]
print(f"Running ensemble on {len(val_images)} validation images...")

processed, skipped = 0, 0
for img_path in val_images:
    try:
        fused_boxes, fused_scores, fused_labels, W, H = run_ensemble_on_image(img_path)
        out_path = f'/content/outputs/ensemble/val_images/{os.path.basename(img_path)}'
        draw_and_save(img_path, fused_boxes, fused_scores,
                      fused_labels, W, H, out_path, class_names)
        processed += 1
    except Exception as e:
        print(f"  Skipped {os.path.basename(img_path)}: {e}")
        skipped += 1

print(f"\nDone. {processed} saved, {skipped} skipped.")
print("Ensemble val images: /content/outputs/ensemble/val_images/")

YOLOv5 weights already at /content/yolov5_best.pt — skipping backup.
Re-cloning YOLOv5...
YOLOv5 re-cloned and deps installed.
Patch applied: True
Cleared 113 cached modules.
YOLOv5 imports successful.
YOLOv8 loaded.


Fusing layers... 
Model summary: 157 layers, 1764577 parameters, 0 gradients, 4.1 GFLOPs


YOLOv5 loaded.
Running ensemble on 40 validation images...

Done. 40 saved, 0 skipped.
Ensemble val images: /content/outputs/ensemble/val_images/


In [22]:
# CELL 9 - Run all 3 models on TEST images and save outputs
import os, glob, shutil
import torch
from ultralytics import YOLO

os.makedirs('/content/outputs/yolov8/test_images', exist_ok=True)
os.makedirs('/content/outputs/yolov5/test_images', exist_ok=True)
os.makedirs('/content/outputs/ensemble/test_images', exist_ok=True)

# ── Find test images ──────────────────────────────────────────────────────────
test_images = glob.glob(
    '/content/datasets/african_small/images/test/**/*.*', recursive=True)
test_images = [p for p in test_images
               if p.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]

# Fallback to val if no test split
if not test_images:
    print("No test split found — using val images as test set.")
    test_images = glob.glob(
        '/content/datasets/african_small/images/val/**/*.*', recursive=True)
    test_images = [p for p in test_images
                   if p.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]

print(f"Found {len(test_images)} test images.")

# ── YOLOv8 test inference ─────────────────────────────────────────────────────
print("Running YOLOv8 on test images...")
model_v8 = YOLO('/content/yolov8_best.pt')
results_v8_test = model_v8.predict(
    source=test_images,
    imgsz=640,
    conf=0.25,
    save=True,
    name='yolov8_test'
)
for img in glob.glob('runs/detect/yolov8_test*/**/*.jpg', recursive=True):
    shutil.copy(img, '/content/outputs/yolov8/test_images/')
for img in glob.glob('runs/detect/yolov8_test*/**/*.png', recursive=True):
    shutil.copy(img, '/content/outputs/yolov8/test_images/')
print(f"YOLOv8 test images saved: /content/outputs/yolov8/test_images/")

# ── YOLOv5 test inference (using local model from Cell 8) ─────────────────────
# model_v5_local is already loaded in Cell 8 — reuse it directly
print("Running YOLOv5 on test images...")
import cv2
import numpy as np
from PIL import Image
import yaml

with open('/content/datasets/african_small/data_custom.yaml') as f:
    cfg = yaml.safe_load(f)
class_names = cfg['names']

for img_path in test_images:
    try:
        b5, s5, l5 = yolov5_predict(model_v5_local, img_path, conf_thr=0.25)
        img_cv = cv2.imread(img_path)
        if img_cv is None:
            img_cv = cv2.cvtColor(
                np.array(Image.open(img_path).convert('RGB')), cv2.COLOR_RGB2BGR)
        h, w = img_cv.shape[:2]
        for box, score, label in zip(b5, s5, l5):
            x1=int(box[0]*w); y1=int(box[1]*h)
            x2=int(box[2]*w); y2=int(box[3]*h)
            cls_name = class_names[label] if label < len(class_names) else str(label)
            cv2.rectangle(img_cv, (x1,y1), (x2,y2), (255,128,0), 2)
            cv2.putText(img_cv, f"{cls_name} {score:.2f}",
                        (x1, max(y1-5,10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,128,0), 2)
        out_path = f'/content/outputs/yolov5/test_images/{os.path.basename(img_path)}'
        cv2.imwrite(out_path, img_cv)
    except Exception as e:
        print(f"  YOLOv5 skipped {os.path.basename(img_path)}: {e}")

print(f"YOLOv5 test images saved: /content/outputs/yolov5/test_images/")

# ── Ensemble test inference (reuse run_ensemble_on_image from Cell 8) ─────────
print("Running Ensemble on test images...")
processed, skipped = 0, 0
for img_path in test_images:
    try:
        fused_boxes, fused_scores, fused_labels, W, H = run_ensemble_on_image(img_path)
        out_path = f'/content/outputs/ensemble/test_images/{os.path.basename(img_path)}'
        draw_and_save(img_path, fused_boxes, fused_scores,
                      fused_labels, W, H, out_path, class_names)
        processed += 1
    except Exception as e:
        print(f"  Ensemble skipped {os.path.basename(img_path)}: {e}")
        skipped += 1

print(f"Ensemble test images saved: /content/outputs/ensemble/test_images/ ({processed} images)")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n── Test inference complete ──")
for folder, label in [
    ('/content/outputs/yolov8/test_images', 'YOLOv8'),
    ('/content/outputs/yolov5/test_images', 'YOLOv5'),
    ('/content/outputs/ensemble/test_images', 'Ensemble'),
]:
    count = len([f for f in glob.glob(os.path.join(folder, '*'))
                 if f.lower().endswith(('.jpg','.png','.jpeg'))])
    print(f"  {label:10s}: {count} images in {folder}")

No test split found — using val images as test set.
Found 40 test images.
Running YOLOv8 on test images...



0: 640x640 (no detections), 322.3ms
1: 640x640 (no detections), 322.3ms
2: 640x640 (no detections), 322.3ms
3: 640x640 (no detections), 322.3ms
4: 640x640 (no detections), 322.3ms
5: 640x640 (no detections), 322.3ms
6: 640x640 (no detections), 322.3ms
7: 640x640 (no detections), 322.3ms
8: 640x640 (no detections), 322.3ms
9: 640x640 (no detections), 322.3ms
10: 640x640 (no detections), 322.3ms
11: 640x640 (no detections), 322.3ms
12: 640x640 (no detections), 322.3ms
13: 640x640 (no detections), 322.3ms
14: 640x640 (no detections), 322.3ms
15: 640x640 (no detections), 322.3ms
16: 640x640 (no detections), 322.3ms
17: 640x640 (no detections), 322.3ms
18: 640x640 (no detections), 322.3ms
19: 640x640 (no detections), 322.3ms
20: 640x640 (no detections), 322.3ms
21: 640x640 (no detections), 322.3ms
22: 640x640 (no detections), 322.3ms
23: 640x640 (no detections), 322.3ms
24: 640x640 

In [23]:
# CELL 10 - Compare metrics and print final file summary
import os, glob

print("=" * 60)
print("         MODEL PERFORMANCE COMPARISON")
print("=" * 60)

# YOLOv8 metrics (captured from Cell 6)
try:
    print(f"YOLOv8  | mAP50: {val_results_v8.box.map50:.4f} | mAP50-95: {val_results_v8.box.map:.4f}")
except:
    print("YOLOv8  | metrics not available (rerun Cell 6)")

# YOLOv5 metrics (captured from Cell 7)
try:
    print(f"YOLOv5  | mAP50: {v5_map50:.4f}           | mAP50-95: {v5_map5095:.4f}")
except:
    print("YOLOv5  | metrics not available (rerun Cell 7)")

# Ensemble: WBF improves precision/recall by reducing duplicate/false boxes
# Show avg confidence improvement as proxy
try:
    v8_confs, ens_confs = [], []
    val_imgs = glob.glob('/content/datasets/african_small/images/val/**/*.*', recursive=True)
    val_imgs = [p for p in val_imgs if p.lower().endswith(('.jpg','.jpeg','.png'))][:20]  # sample 20

    import numpy as np
    from ultralytics import YOLO
    model_v8_check = YOLO('/content/yolov8_best.pt')

    for ip in val_imgs:
        r = model_v8_check.predict(source=ip, imgsz=640, conf=0.25, verbose=False)[0]
        if r.boxes is not None and len(r.boxes):
            v8_confs.extend([float(b.conf.cpu().numpy()[0]) for b in r.boxes])
        fb, fs, fl, W, H = run_ensemble_on_image(ip)
        ens_confs.extend(fs.tolist() if hasattr(fs,'tolist') else list(fs))

    print(f"\nEnsemble vs YOLOv8 (sample of 20 val images):")
    print(f"  YOLOv8  avg confidence : {np.mean(v8_confs):.4f} | boxes: {len(v8_confs)}")
    print(f"  Ensemble avg confidence: {np.mean(ens_confs):.4f} | boxes: {len(ens_confs)}")
    print(f"  (WBF reduces duplicate boxes while keeping high-confidence detections)")
except Exception as e:
    print(f"Ensemble comparison skipped: {e}")

print("\n" + "=" * 60)
print("              OUTPUT FILES SUMMARY")
print("=" * 60)

files_to_check = {
    "YOLOv8 weights (.pt)"      : '/content/yolov8_best.pt',
    "YOLOv5 weights (.pt)"      : '/content/yolov5_best.pt',
    "YOLOv8 val images"         : '/content/outputs/yolov8/val_images',
    "YOLOv5 val images"         : '/content/outputs/yolov5/val_images',
    "YOLOv8 test images"        : '/content/outputs/yolov8/test_images',
    "YOLOv5 test images"        : '/content/outputs/yolov5/test_images',
    "Ensemble val images"       : '/content/outputs/ensemble/val_images',
    "Ensemble test images"      : '/content/outputs/ensemble/test_images',
}

for label, path in files_to_check.items():
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1e6
        print(f"  ✅ {label:30s} {path} ({size:.1f} MB)")
    elif os.path.isdir(path):
        count = len([f for f in glob.glob(os.path.join(path,'*'))
                     if f.lower().endswith(('.jpg','.png','.jpeg','.pt'))])
        print(f"  ✅ {label:30s} {path} ({count} files)")
    else:
        print(f"  ❌ {label:30s} NOT FOUND")

# ── Zip everything for easy download ─────────────────────────────────────────
import shutil
shutil.make_archive('/content/submission', 'zip', '/content/outputs')
shutil.copy('/content/yolov8_best.pt', '/content/outputs/yolov8_best.pt')
shutil.copy('/content/yolov5_best.pt', '/content/outputs/yolov5_best.pt')

print("\n📦 All outputs zipped to: /content/submission.zip")
print("   Download via Files panel on the left in Colab.")

         MODEL PERFORMANCE COMPARISON
YOLOv8  | mAP50: 0.2155 | mAP50-95: 0.1530
YOLOv5  | mAP50: 0.0000           | mAP50-95: 0.0000

Ensemble vs YOLOv8 (sample of 20 val images):
  YOLOv8  avg confidence : nan | boxes: 0
  Ensemble avg confidence: 0.2763 | boxes: 3
  (WBF reduces duplicate boxes while keeping high-confidence detections)

              OUTPUT FILES SUMMARY
  ✅ YOLOv8 weights (.pt)           /content/yolov8_best.pt (6.2 MB)
  ✅ YOLOv5 weights (.pt)           /content/yolov5_best.pt (14.7 MB)
  ✅ YOLOv8 val images              /content/outputs/yolov8/val_images (12 files)
  ✅ YOLOv5 val images              /content/outputs/yolov5/val_images (9 files)
  ✅ YOLOv8 test images             /content/outputs/yolov8/test_images (40 files)
  ✅ YOLOv5 test images             /content/outputs/yolov5/test_images (40 files)
  ✅ Ensemble val images            /content/outputs/ensemble/val_images (40 files)
  ✅ Ensemble test images           /content/outputs/ensemble/test_images (40 f

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



📦 All outputs zipped to: /content/submission.zip
   Download via Files panel on the left in Colab.
